# Experiment Comparison: Cross-Run Analysis

This notebook compares metrics across multiple experiment runs to
identify the impact of different configurations (topology, model,
memory settings, etc.).

In [ ]:
from data_loader import (
    load_experiment,
    extract_action_events,
    extract_movement_events,
    extract_reflection_events,
    compute_location_transition_matrix,
    compute_interaction_matrix,
)

from collections import Counter, defaultdict
import json

## 1. Define Experiments to Compare

Replace these with your actual experiment IDs.

In [ ]:
experiments = {
    "ring_default": "exp_abc123",    # <-- change
    "small_world": "exp_def456",     # <-- change
    "with_goals": "exp_ghi789",     # <-- change
}

## 2. Load All Experiment Data

In [ ]:
experiment_data: dict[str, dict] = {}

for label, eid in experiments.items():
    try:
        events = load_experiment(eid)
        experiment_data[label] = {
            "events": events,
            "actions": extract_action_events(events),
            "movements": extract_movement_events(events),
            "reflections": extract_reflection_events(events),
        }
        print(f"  {label} ({eid}): {len(events)} events")
    except FileNotFoundError as e:
        print(f"  {label} ({eid}): NOT FOUND - {e}")

## 3. Summary Statistics Comparison

In [ ]:
def compute_stats(label: str, data: dict) -> dict:
    events = data["events"]
    type_counts = Counter(e.get("event_type", "unknown") for e in events)
    agents = sorted(set(e.get("agent_id", "") for e in events if e.get("agent_id")))
    steps = [e.get("step", 0) for e in events]
    
    transition_matrix = compute_location_transition_matrix(events)
    total_transitions = sum(
        sum(dests.values()) for dests in transition_matrix.values()
    )
    
    interaction_matrix = compute_interaction_matrix(events)
    total_interactions = sum(
        sum(partners.values()) for partners in interaction_matrix.values()
    )
    
    return {
        "Label": label,
        "Total Events": len(events),
        "Total Steps": max(steps) if steps else 0,
        "Agents": len(agents),
        "Actions": type_counts.get("action", 0),
        "Movements": type_counts.get("movement", 0),
        "Reflections": type_counts.get("reflection", 0),
        "Daily Plans": type_counts.get("daily_plan", 0),
        "Hourly Plans": type_counts.get("hourly_plan", 0),
        "Total Transitions": total_transitions,
        "Total Interactions": total_interactions,
    }

rows = [compute_stats(label, data) for label, data in experiment_data.items()]

# Print comparison table
if rows:
    cols = list(rows[0].keys())
    col_widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) + 2 for c in cols}
    
    header = " | ".join(c.rjust(col_widths[c]) for c in cols)
    print(header)
    print("-" * len(header))
    for row in rows:
        line = " | ".join(str(row[c]).rjust(col_widths[c]) for c in cols)
        print(line)

## 4. Movement Patterns Comparison

Compare how agents move across different experiment configurations.

In [ ]:
print("Top 5 transitions per experiment:")
print()

for label, data in experiment_data.items():
    tm = compute_location_transition_matrix(data["events"])
    all_trans = []
    for src, dests in tm.items():
        for dst, count in dests.items():
            all_trans.append((src, dst, count))
    all_trans.sort(key=lambda x: -x[2])
    
    print(f"  {label}:")
    for src, dst, count in all_trans[:5]:
        print(f"    {src} -> {dst}: {count}")
    print()

## 5. Agent Activity Distribution

Compare action distribution across agents between experiments.

In [ ]:
for label, data in experiment_data.items():
    action_counts = Counter(e.get("agent_id") for e in data["actions"])
    if not action_counts:
        continue
    
    total = sum(action_counts.values())
    print(f"  {label} (total actions: {total}):")
    for agent, count in action_counts.most_common():
        pct = 100 * count / total if total > 0 else 0
        print(f"    {agent}: {count} ({pct:.1f}%)")
    print()

## 6. Reflection Frequency Comparison

In [ ]:
for label, data in experiment_data.items():
    refs = data["reflections"]
    trigger_counts = Counter()
    for e in refs:
        d = e.get("data", {})
        if isinstance(d, dict):
            trigger_counts[d.get("trigger", "unknown")] += 1
    
    print(f"  {label}: {len(refs)} reflections")
    for trigger, count in trigger_counts.most_common():
        print(f"    {trigger}: {count}")
    print()

## 7. Key Findings Template

Fill in your observations after running the analysis above.

In [ ]:
findings = """
Key Findings:

1. [Topology Impact]:
   - Ring vs Small-world: ___________

2. [Goal Impact]:
   - With goals vs without: ___________

3. [Agent Diversity]:
   - Most/least active agents: ___________

4. [Movement Patterns]:
   - Dominant transitions: ___________

5. [Unexpected Observations]:
   - ___________
"""
print(findings)